You built an amazing model. It works perfectly on your laptop. Now a colleague in another country needs to use it on their data. How do you get it to them -- reliably, reproducibly, and at scale? That is the MLOps problem, and this notebook solves it.


# Module 16 — MLOps for Protein ML

## TL;DR — Plain English

**What is MLOps?** MLOps bridges the gap between "my notebook runs" and "this runs reliably in production for a team." It is the set of practices that make ML systems reproducible, auditable, and deployable at scale.

### The Gap: Notebook vs Production

| Notebook | Production |
|----------|------------|
| Run once on your laptop | Runs 1000x/day on servers |
| You remember what you did | Others need to reproduce your results |
| Any result is fine | Wrong predictions cost money or patient health |
| Dependencies informal | Pinned versions, containers |
| No monitoring | Alerts when model degrades |

### Five Pillars of MLOps

1. **Reproducibility** — Same code + same data + same seed → same results. Requires: config files, seed management, data checksums, pinned dependencies.
2. **Experiment Tracking** — Which hyperparameters gave the best AUC? Which protein family was hardest? Tools: MLflow, Weights & Biases, TensorBoard.
3. **Model Serving** — An API endpoint that takes a protein sequence and returns a prediction. Tools: FastAPI + uvicorn, TorchServe, HuggingFace Inference Endpoints.
4. **Monitoring** — Did my model's accuracy drop? Are the input sequences from a different distribution than training? Tools: Prometheus + Grafana, custom KS-test monitors.
5. **CI/CD** — Every code change is automatically tested and deployed. Tools: GitHub Actions, Jenkins, DVC.

### Why Protein ML Specifically Needs MLOps

- **Expensive GPU runs**: AlphaFold3 training costs ~\$1M. You cannot redo experiments carelessly — every run must be logged.
- **Multiple model versions**: AF2, AF3, ESM2-8M, ESM2-15B, ESMFold, OpenFold — tracking which version predicted which structure is critical for wet lab follow-up.
- **Wet lab validation cycles**: A computational prediction today is validated experimentally in 6–12 weeks. You need an audit trail: which model, which weights, which input → which prediction → which experiment.
- **Regulatory pressure**: Clinical applications (drug discovery) require model cards, uncertainty estimates, and reproducibility documentation.
- **Distribution shift**: Training on UniProt; production sees novel protein families, pathogen variants, synthetic proteins — monitoring is essential.

### What You Will Be Able to Do After This Notebook

- Track any experiment with MLflow (parameters, metrics, artifacts)
- Write a fully reproducible training script (seeds, configs, checksums)
- Checkpoint models with metadata and auto-keep-top-k
- Build a FastAPI REST endpoint for model serving
- Write a Dockerfile and docker-compose for GPU-enabled ML services
- Detect data drift with the Kolmogorov-Smirnov test
- Design a GitHub Actions CI/CD pipeline for an ML model
- Answer senior-level MLOps interview questions

## 🛠️ MLOps Concepts for Beginners

| Term | Plain English |
|------|--------------|
| **MLflow** | Dashboard for tracking ML experiments — logs hyperparameters, metrics, model files, and lets you compare runs |
| **FastAPI** | Python library to build a web API in <20 lines — turns your model into a URL you can call from anywhere |
| **Docker** | Packages your code + Python version + dependencies into a portable container that runs identically everywhere |
| **DVC** | Git for large data files — tracks which dataset version trained which model, enables reproducibility |
| **endpoint** | A URL that accepts input and returns output (e.g. POST /predict sends a mutation, gets back ΔΔG) |
| **async def** | Non-blocking function — FastAPI uses this to handle thousands of requests concurrently |
| **Pydantic model** | A Python class with automatic input validation — ensures API gets the right types before running your model |
| **drift detection** | Monitoring whether new production inputs look statistically different from training data |
| **Mahalanobis distance** | Statistical distance that accounts for correlations between features — better than Euclidean for multivariate drift |
| **CI/CD** | Automated pipeline: every code push triggers tests → build → deploy (GitHub Actions, etc.) |
| **experiment run** | One training session with specific hyperparameters — MLflow logs everything about it |
| **model registry** | Catalogue of trained model versions with metadata — MLflow provides one for free |

## Beginner Teaching Frame

**Who should fully work through this notebook:** students who already know how to train models and now need to think like an ML systems engineer.

**How to study it on a first pass:**
- focus on reproducibility, checkpointing, serving, and monitoring
- treat specific tools as examples of deeper systems ideas
- ask what can break after training is over

**Common traps:** thinking MLOps is just Docker, or treating deployment as separate from model quality and data quality.


## 📖 Prerequisites & Cross-References

**Before this notebook, you should be comfortable with:**
- **PyTorch model training** — training loop, saving/loading models. *Review: `05_deep_learning_finetuning/01_dl_and_finetuning.ipynb`*
- **Python basics** — classes, functions, decorators. *Review: `00_python_ml_basics/01_python_core_for_bioinformatics.ipynb`*

**Quick recap:**
- **MLflow** — experiment tracking dashboard for hyperparameters, metrics, and model artifacts
- **FastAPI** — Python web framework to turn models into API endpoints
- **Docker** — packages code + dependencies into a portable container
- **DVC** — "git for data" — tracks which dataset trained which model

## Canonical Resource Upgrade

Strong references here:
- [Made With ML](https://madewithml.com/)
- [Full Stack Deep Learning](https://fullstackdeeplearning.com/)
- [MLflow docs](https://mlflow.org/docs/latest/index.html)
- [FastAPI tutorial](https://fastapi.tiangolo.com/tutorial/)


## Cross-References

### Prerequisites
- **00/06 PyTorch Fundamentals** — model classes, state_dict, optimizers (used in checkpointing section)
- **00/02 ML Fundamentals** — metrics (AUC, accuracy), train/val/test splits, pipelines (assumed throughout)

### Directly Related Modules
- **10/01 Protein Structure Fine-Tuning** — the LoRA-fine-tuned ESM2 model built there is exactly what you would deploy using this notebook's patterns
- **07/01 AlphaFold3 Core** — production AlphaFold3 inference is the canonical example of large model serving; the monitoring patterns here apply to AF3 predictions
- **11/01 Membrane Protein Dynamics** — LoRA adapters trained per protein family can be served using the multi-adapter pattern described here
- **15/01 Self-Supervised Learning** — ESM2 pre-training pipelines require MLflow tracking at scale; reproducibility is critical

### Position in Curriculum

```
Modules 00-15: Learn the science and build the models
       |
       v
Module 16 (this notebook): How to ship everything you built
```

This is the **final notebook** in the curriculum. It teaches the engineering practices that turn research models into production systems — the skill that separates senior ML engineers from researchers at companies like computational biology ML teams and structural biology research labs.

> **Why this code:** This defines the training loop that iteratively improves the model by learning from data.

In [ ]:
import os                       # os: file paths, environment variables, directory operations
import hashlib                   # hashlib: SHA-256/MD5 fingerprints for data and config versioning
import json                      # json: serialize configs and experiment metadata to files
import random                    # random: Python's built-in random number generator (for seeding)
import numpy as np              # numpy: numerical arrays and math operations
import torch                    # PyTorch: the main deep learning library

# Reproducibility: seeds, configs, checksums
# --- set_all_seeds() ---
def set_all_seeds(seed=42):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"All seeds set to {seed}")

set_all_seeds(42)

# Config management
config = {
    "model": {"d_model": 256, "n_heads": 8, "n_layers": 4, "dropout": 0.1},
    "training": {"lr": 1e-4, "batch_size": 32, "epochs": 100, "seed": 42},
    "data": {"dataset": "skempi_v2", "train_split": 0.8, "normalize": True},
    "lora": {"rank": 8, "alpha": 16, "dropout": 0.05},
}
# Compute config hash for experiment tracking
config_hash = hashlib.md5(json.dumps(config, sort_keys=True).encode()).hexdigest()[:8]
print(f"\nConfig hash: {config_hash}")
print(f"Config: {json.dumps(config, indent=2)[:200]}...")

# Data checksum
# --- compute_file_checksum() ---
def compute_file_checksum(filepath):
    """
    Plain English: reads a file and computes a short SHA-256 fingerprint — used to verify that the exact same data was used across experiments.

    Args:
        filepath: path to any binary or text file
    Returns:
        str — first 16 hex characters of the SHA-256 hash
    """
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        h.update(f.read())
    # Return result
    return h.hexdigest()[:16]

# Simulate checksum check
print(f"\nReproducibility checklist:")
print(f"  ✓ Seed: {config['training']['seed']}")
print(f"  ✓ Config hash: {config_hash}")
print(f"  ✓ CUDA deterministic: {torch.backends.cudnn.deterministic}")
print(f"  ✓ torch version: {torch.__version__}")

> 💡 **Analogy:** Think of this concept like a recipe — each step transforms the ingredients (data) into something more useful, and the order matters.

> ⚠️ **Common Mistake:** Forgetting `model.eval()` before inference (dropout and batch norm stay in training mode)
>
> **Wrong:** `# model.train() is the default after training
pred = model(x_test)  # dropout randomly zeros neurons, predictions are noisy!`
> **Right:** `model.eval()
with torch.no_grad():
    pred = model(x_test)  # dropout disabled, batch norm uses running stats`
> **Why:** In training mode, dropout randomly zeros neurons and batch norm uses batch statistics. At inference, you want deterministic predictions using the learned parameters.

> **Expected output:** Reproducibility setup: seed value, config hash, truncated config JSON, and a checklist confirming deterministic CUDA, torch version, and seed are set  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

### 📖 Reading Guide — FastAPI Protein Prediction Service

`app = FastAPI()`
→ *Create a web application. FastAPI automatically generates interactive documentation at /docs.*

`@app.post('/predict')`
→ *Decorator: this function handles POST requests to the /predict URL. POST = sending data (the sequence) to the server.*

`async def predict(request: PredictionRequest):`
→ *async = the function can handle multiple requests simultaneously without blocking. PredictionRequest validates the input automatically.*

`embedding = model.encode(request.sequence)`
→ *Convert the protein sequence to a vector embedding. This is the slow step — ESM-2 inference.*

`return {'ddg': float(prediction), 'confidence': float(confidence)}`
→ *Return JSON. FastAPI converts the Python dict to JSON automatically.*



> **Building up gradually.** The class below might look intimidating — but every piece is something you've already seen. Let's break it down:
>
> - Lines 28-32: Sets up the layers and parameters
> - Lines 33-64: The forward pass — how data flows through
>
> Read the breakdown first, THEN read the code.

> **Let's trace through this step by step.** Before you run the cell below, read each line and predict what it does. Then run it and check — did you predict correctly?

In [ ]:
# --- Imports ---
import mlflow                   # mlflow: experiment tracking — logs params, metrics, and model artifacts
import torch                    # PyTorch: the main deep learning library
import torch.nn as nn           # nn: neural network building blocks (layers, losses)
import numpy as np              # numpy: numerical arrays and math operations

# MLflow experiment tracking
print("MLflow Experiment Tracking for Protein ML")
print("=" * 50)

# Set up MLflow (in-memory for demo)
mlflow.set_tracking_uri("file:///tmp/mlruns")
mlflow.set_experiment("protein_ddg_prediction")

# --- SimpleModel class definition ---
class SimpleModel(nn.Module):
    """Minimal protein property prediction network for MLOps demonstration.
    Architecture: Linear(input_dim → hidden) → ReLU → Linear(hidden → 1)

    Args (in __init__):
        input_dim: number of input features (default 20 — one per amino acid type)
        hidden: hidden layer size (default 64)
    Input (forward):
        x: float tensor (batch, input_dim) — amino acid composition or embedding
    Output (forward):
        tensor (batch, 1) — predicted property (e.g. ΔΔG)"""
    # --- __init__() function ---
    def __init__(self, input_dim=20, hidden=64):
        super().__init__()
        # Set net
        self.net = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    # --- forward() function ---
    def forward(self, x): return self.net(x)

torch.manual_seed(42)
model = SimpleModel()
X = torch.randn(100, 20); y = torch.randn(100, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Log experiment
with mlflow.start_run(run_name="baseline_ridge_v1") as run:
    # Log parameters
    mlflow.log_params({
        "model_type": "SimpleModel",
        "hidden_dim": 64,
        "lr": 1e-3,
        "seed": 42,
    })

    # Training loop with logging
    for epoch in range(5):
        pred = model(X)
        loss = nn.MSELoss()(pred, y)
        # Optimizer update
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        pearson_r = np.corrcoef(pred.detach().numpy().flatten(), y.numpy().flatten())[0,1]
        mlflow.log_metrics({"train_loss": loss.item(), "pearson_r": pearson_r}, step=epoch)

    # Log model artifact
    mlflow.pytorch.log_model(model, "model")
    print(f"Run ID: {run.info.run_id[:8]}...")
    print(f"Final loss: {loss.item():.4f}")
    print(f"Final Pearson r: {pearson_r:.4f}")
    print(f"Logged to: /tmp/mlruns")

## 💡 Worked Example: Simplest Possible API

Before the full FastAPI setup, here's the concept in pseudocode:

```python
# An API is just: receive input → process → return output
# In Flask (simpler):
@app.route('/predict', methods=['POST'])
def predict():
    data = request.json          # step 1: receive
    result = model.predict(data) # step 2: process
    return {'prediction': result} # step 3: return

# FastAPI adds: automatic input validation, async support, auto-generated docs
```

The next cell builds this with FastAPI + Pydantic for production quality.

### MLflow Checkpointing — Save and Resume Training

This cell implements `save_checkpoint` and `load_checkpoint` to persist the full training state: model weights, optimizer state (including Adam momentum buffers), epoch number, and metadata.

**Why save the optimizer state?** Adam keeps a running mean and variance of gradients for each parameter. Without it, resuming training would effectively reset momentum — causing a temporary spike in loss before the optimizer "warms up" again.

In [ ]:
import torch                    # PyTorch: the main deep learning library
import torch.nn as nn           # nn: neural network building blocks (layers, losses)
import os                       # os: file I/O and path utilities

# Checkpointing — save/load training state
def save_checkpoint(model, optimizer, epoch, loss, path, metadata=None):
    """Save full training checkpoint (model + optimizer state)."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
        'metadata': metadata or {},
    }
    torch.save(checkpoint, path)
    size_mb = os.path.getsize(path) / 1e6
    # Output
    print(f"Checkpoint saved: epoch={epoch}, loss={loss:.4f}, size={size_mb:.2f}MB")

# Define load_checkpoint()
def load_checkpoint(path, model, optimizer=None):
    """Resume training from checkpoint."""
    ckpt = torch.load(path, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    # Conditional check
    if optimizer and 'optimizer_state_dict' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    # Output
    print(f"Loaded checkpoint: epoch={ckpt['epoch']}, loss={ckpt['loss']:.4f}")
    return ckpt['epoch'], ckpt['loss']

# Demo
torch.manual_seed(42)
# Sequential layer
model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Train for 3 steps
X = torch.randn(32, 20); y = torch.randn(32, 1)
# Iterate
for step in range(3):
    loss = nn.MSELoss()(model(X), y)
    # Backpropagate gradients
    optimizer.zero_grad(); loss.backward(); optimizer.step()

# Configure optimizer
save_checkpoint(model, optimizer, epoch=3, loss=loss.item(),
                path='/tmp/model_epoch3.pt',
                metadata={'dataset': 'skempi_v2', 'metric': 'mse'})

# Simulate resuming
model2 = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 1))
opt2 = torch.optim.Adam(model2.parameters(), lr=1e-3)
epoch_resumed, loss_resumed = load_checkpoint('/tmp/model_epoch3.pt', model2, opt2)
# Output
print(f"Resumed from epoch {epoch_resumed}")

### FastAPI Endpoint — Serving ΔΔG Predictions

This cell defines a minimal REST API using FastAPI. It exposes two endpoints:
- `GET /health` — liveness probe (returns "ok")
- `POST /predict` — accepts a mutation description and returns predicted ΔΔG

**Pydantic input validation:** FastAPI uses the `MutationRequest` schema to automatically reject malformed requests before they reach the model — no manual input checking needed.

**To run in production:** `uvicorn main:app --host 0.0.0.0 --port 8000`

In [ ]:
# --- Imports ---
from fastapi import FastAPI     # FastAPI: web framework for building API endpoints quickly
from pydantic import BaseModel  # BaseModel: define typed data schemas with automatic validation
import torch                    # PyTorch: the main deep learning library
import torch.nn as nn           # nn: neural network building blocks
import numpy as np              # numpy: numerical arrays and math operations
import uvicorn                  # uvicorn: ASGI server that runs FastAPI apps
import threading                # threading: run server in background thread for demo purposes

# FastAPI serving for protein ΔΔG prediction
app = FastAPI(title="Protein ΔΔG Predictor API")

# --- MutationRequest class definition ---
class MutationRequest(BaseModel):
    """Pydantic schema for the /predict API endpoint request body.
    FastAPI automatically validates that incoming JSON matches this schema.

    Fields:
        sequence: protein sequence string (one-letter amino acid codes)
        position: 0-indexed residue position to mutate
        mutation: single-letter amino acid code to substitute at that position"""
    sequence: str
    position: int
    wildtype: str
    mutant: str

# --- DDGResponse class definition ---
class DDGResponse(BaseModel):
    """Pydantic schema for the /predict API endpoint response body.
    FastAPI serializes the returned object to JSON automatically.

    Fields:
        ddg: predicted ΔΔG in kcal/mol (negative = stabilizing mutation)
        confidence: model confidence score (0.0–1.0)
        warning: optional message if input looks unusual"""
    ddg: float
    confidence: float
    interpretation: str

# Load model (simulated)
torch.manual_seed(42)
# Compute predictor via nn.Sequential
predictor = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 1))

@app.get("/health")
# --- health() function ---
def health():
    """Health-check endpoint — GET /health.
    Returns service status and model version.
    Used by load balancers and orchestrators to verify the service is alive."""
    # Return result
    return {"status": "ok", "model": "ddg_predictor_v1"}

@app.post("/predict", response_model=DDGResponse)
# --- predict() function ---
def predict(request: MutationRequest):
    """Prediction endpoint — POST /predict.
    Accepts a MutationRequest JSON body, runs inference through SimpleModel,
    and returns a DDGResponse with predicted ΔΔG and confidence score.

    Args (via HTTP JSON body — validated by MutationRequest):
        sequence: protein sequence string
        position: 0-indexed residue position to mutate
        mutation: single-letter amino acid to substitute
    Returns:
        DDGResponse — ddg (kcal/mol), confidence (0–1), optional warning"""
    # Simulate feature extraction
    features = torch.zeros(1, 20)
    # Update features entry
    features[0, hash(request.wildtype) % 20] = 1.0
    # Update features entry
    features[0, hash(request.mutant) % 20] -= 0.5

    # Context manager
    with torch.no_grad():
        # Compute ddg via predictor
        ddg = predictor(features).item()

    # Calculate interpretation
    interpretation = "Stabilizing" if ddg < -0.5 else "Destabilizing" if ddg > 0.5 else "Neutral"
    return DDGResponse(ddg=round(ddg, 3), confidence=0.75, interpretation=interpretation)

# Demo: test the API endpoint logic directly
req = MutationRequest(sequence="ACDEFGHIKLM", position=5, wildtype="A", mutant="V")
result = predict(req)
print(f"API Response:")
print(f"  ΔΔG: {result.ddg} kcal/mol")
print(f"  Confidence: {result.confidence}")
print(f"  Interpretation: {result.interpretation}")
print()
print("Deployment: uvicorn main:app --host 0.0.0.0 --port 8000")
print("Docker: FROM python:3.10 + pip install fastapi uvicorn torch + COPY + CMD")

### Data Drift Monitoring

Once your model is deployed, the real-world inputs may gradually shift away from the training distribution — a phenomenon called **data drift**. For protein ML, this happens when you start seeing mutations from different protein families than those in SKEMPI.

This cell implements a `DriftMonitor` class using the **Kolmogorov-Smirnov (KS) test** to compare each feature's distribution between training and production. A p-value below 0.05 triggers a drift alert.

In [ ]:
import numpy as np              # numpy: numerical arrays and math operations
from scipy import stats         # stats: scientific statistics (KS test, t-test, etc.)

# Data drift monitoring for protein ML models
print("Production Drift Monitoring for Protein ML")
print("=" * 60)
print()

# Process
class DriftMonitor:
    """
    Plain English: watches for statistical drift between training-time and production-time input distributions — alerts you when the model is seeing inputs it was never trained on.

    Method used: Kolmogorov-Smirnov (KS) test — compares two distributions without assuming they are Gaussian.
    A p-value below the threshold means the distributions are significantly different (drift detected).

    Args (constructor):
        reference_data: 2D numpy array of shape (n_samples, n_features) from training set
        feature_names:  optional list of feature name strings for readable reports
        threshold:      p-value cutoff for drift detection (default 0.05)
    """
    def __init__(self, reference_data, feature_names=None, threshold=0.05):
        self.reference = reference_data
        # Compute feature_names
        self.feature_names = feature_names or [f'feat_{i}' for i in range(reference_data.shape[1])]
        self.threshold = threshold

    def check_drift(self, new_data):
        """
        Plain English: runs a KS test on each feature comparing reference (training) vs new (production) distributions.

        Args:
            new_data: 2D numpy array of shape (m_samples, n_features) from production
        Returns:
            list of dicts — one per feature with keys: feature, ks_stat, p_value, drifted (bool)
        """
        # Compute n_features
        n_features = self.reference.shape[1]
        drift_report = []
        for i in range(n_features):
            ks_stat, p_val = stats.ks_2samp(self.reference[:, i], new_data[:, i])
            # Compute drifted
            drifted = p_val < self.threshold
            drift_report.append({
                'feature': self.feature_names[i],
                'ks_stat': round(ks_stat, 4),
                # Process
                'p_value': round(p_val, 4),
                'drifted': drifted
            })
        return drift_report

# Process
np.random.seed(42)
# Training distribution (SKEMPI mutations)
ref_data = np.random.randn(1000, 10)
# Production data: slight drift (new mutations from different proteins)
prod_data = np.random.randn(200, 10) + np.array([0.5,0,0,0.3,0,0,0,0.2,0,0])

monitor = DriftMonitor(ref_data, threshold=0.05)
report = monitor.check_drift(prod_data)
# Compute n_drifted
n_drifted = sum(r['drifted'] for r in report)
print(f"Features with drift: {n_drifted}/{len(report)}")
print()
for r in report:
    # Compute flag
    flag = "⚠ DRIFT" if r['drifted'] else "OK"
    print(f"  {r['feature']}: KS={r['ks_stat']:.4f} p={r['p_value']:.4f} {flag}")
print()
print("Action when drift detected:")
# Print results
print("  1. Alert on-call ML engineer")
print("  2. Log incoming data for retraining")
print("  3. Consider rolling back to previous model version")

### CI/CD Pipeline for ML Models

A CI/CD (Continuous Integration / Continuous Deployment) pipeline is an automated system that runs tests, validates model performance, and deploys new versions whenever code is merged.

For ML models, CI/CD adds a critical extra step between "tests pass" and "deploy": **model validation** — the new model must hit a minimum performance threshold on a held-out dataset before it is allowed into production.

This cell prints a representative pipeline config covering code quality checks, unit tests, model validation, Docker build/test, and Kubernetes rollout.

In [ ]:
import subprocess  # subprocess: run shell commands from Python (used for CI/CD demo output)
import json       # json: read and write JSON-structured data

# CI/CD pipeline for ML models
print("CI/CD Pipeline for Protein ML")
print("=" * 60)
print()

# Compute pipeline_config
pipeline_config = {
    "name": "protein-ml-ci",
    "trigger": ["push to main", "pull request"],
    "stages": [
        # Process
        {
            "name": "1. Code Quality",
            "steps": ["flake8 src/", "black --check src/", "mypy src/"]
        },
        # Process
        {
            "name": "2. Unit Tests",
            "steps": ["pytest tests/ -v --tb=short", "pytest --cov=src --cov-report=xml"]
        },
        # Process
        {
            "name": "3. Model Validation",
            "steps": [
                "python validate_model.py --dataset val_set.csv --metric pearson_r",
                # Compute "assert pearson_r >
                "assert pearson_r >= 0.65  # performance threshold"
            ]
        },
        {
            # Process
            "name": "4. Integration Tests",
            "steps": [
                "docker build -t protein-ml:test .",
                "docker run protein-ml:test python -m pytest tests/integration/",
            # Process
            ]
        },
        {
            "name": "5. Deploy (on merge to main)",
            # Process
            "steps": [
                "docker push registry/protein-ml:latest",
                "kubectl set image deployment/ddg-api ddg-api=protein-ml:${SHA}",
                "kubectl rollout status deployment/ddg-api"
            # Process
            ]
        },
    ]
}

# Process
for stage in pipeline_config["stages"]:
    print(f"Stage: {stage['name']}")
    for step in stage["steps"]:
        print(f"  $ {step}")
    # Print results
    print()

print("Best practices:")
practices = [
    "Never push model weights to git (use DVC or MLflow)",
    # Process
    "Pin dependency versions (requirements.txt + pip-compile)",
    "Test with small model variant in CI (fast, no GPU needed)",
    "Gate deployment on validation metric threshold",
    "Blue-green deployment: route 10% traffic to new model first",
# Process
]
for p in practices:
    print(f"  • {p}")

### Gradient Accumulation — Simulating Large Batches on Limited GPU

Protein structure models are large — a single protein with 300 residues may barely fit in one GPU batch. Gradient accumulation is the standard solution: run several small micro-batches, sum the gradients, then update weights once.

**Effective batch size** = `micro_batch_size × accumulation_steps`. Here we use 8 × 4 = 32.

**Why does batch size matter?** Larger batches give more stable gradient estimates, which typically leads to smoother training curves and better final performance.

In [ ]:
import numpy as np              # numpy: numerical arrays and math operations
import torch                    # PyTorch: the main deep learning library
import torch.nn as nn           # nn: neural network building blocks (layers, losses)

# Gradient accumulation for large-batch training on limited GPU
print("Gradient Accumulation: Simulate Large Batches")
print("=" * 60)

torch.manual_seed(42)
# Compute model
model = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# Effective batch size = micro_batch * accumulation_steps
micro_batch = 8
accum_steps = 4
effective_batch = micro_batch * accum_steps
# Print results
print(f"Micro batch: {micro_batch}, Accumulation: {accum_steps}×")
print(f"Effective batch size: {effective_batch} (simulates GPU with {effective_batch}× more memory)")

losses = []
for step in range(3):  # 3 "effective" gradient steps
    # Process
    optimizer.zero_grad()
    total_loss = 0
    for micro_step in range(accum_steps):
        X = torch.randn(micro_batch, 256)
        # Compute y
        y = torch.randn(micro_batch, 1)
        pred = model(X)
        loss = nn.MSELoss()(pred, y) / accum_steps  # normalize by accum steps
        loss.backward()  # accumulate gradients
        # Compute total_loss +
        total_loss += loss.item()
    # Clip gradients and update
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    losses.append(total_loss)
    # Compute print(f"  Step {step+1}: loss
    print(f"  Step {step+1}: loss={total_loss:.4f}")

print(f"\nGradient accumulation enables training with effective batch={effective_batch}")
print("Critical for Pairformer fine-tuning where single protein barely fits on GPU")

## Interview Q&A

---

**Q: What is train-serve skew and why is it dangerous?**

A: Train-serve skew occurs when features are computed differently at training time versus serving time. Example: training uses `np.mean(all_sequences)` over the full dataset; serving computes mean over a single sample — giving a different normalisation. The model sees out-of-distribution inputs even for normal sequences, causing silent accuracy degradation (no error, just wrong predictions). Solution: a **feature store** (Feast, Tecton, AWS SageMaker Feature Store) that centralises feature computation so both training and serving call identical code.

---

**Q: How would you detect if your protein stability model has degraded in production?**

A: Use a layered monitoring strategy:
1. **Input drift**: KS test on amino acid frequencies, sequence length histogram, protein family distribution (cluster with ESM2 embeddings).
2. **Output drift**: histogram of predicted ΔΔG scores shifts toward extreme values.
3. **Shadow testing**: run both old and new model; flag cases where predictions disagree by >1 kcal/mol.
4. **Slice-based evaluation**: track AUC separately per protein family (soluble vs membrane vs IDR).
5. **Labelled sentinel set**: 50 proteins with known experimental ΔΔG values; evaluate weekly.

---

**Q: MLflow vs Weights & Biases vs TensorBoard — when to use each?**

| Tool | Best For | Limitations |
|------|----------|-------------|
| TensorBoard | Quick training visualisation; built into PyTorch/TF | Metrics only; no artifact management |
| MLflow | Full experiment lifecycle (params, metrics, artifacts, model registry); self-hostable | UI less polished than W&B |
| W&B | Best UX; team collaboration; protein structure visualisation | Paid at scale; data leaves your servers |

For bioinformatics at computational biology ML teams / structural biology research labs: **MLflow preferred** — self-hosted, protein sequence data stays on-prem, integrates with SLURM-based GPU clusters.

---

**Q: How do you serve a 15B parameter model (ESM2-15B) at production latency?**

A:
1. **Quantisation**: INT8 via `bitsandbytes` or `torch.quantization` → 4× memory reduction, ~2× speedup.
2. **vLLM / TGI**: paged KV-cache attention enables continuous batching → higher throughput.
3. **Model parallelism**: tensor parallelism across multiple A100s using `deepspeed` or `megatron`.
4. **LoRA adapter serving**: keep base model once in memory, swap per-task LoRA adapters at request time (adds ~1 ms).
5. **Distillation**: train ESM2-8M (43M params) to mimic ESM2-15B via knowledge distillation → 350× smaller.

---

**Q: What is a feature store and when does protein ML need one?**

A: A feature store is a centralised system that computes, stores, and serves ML features consistently. Both the training pipeline and serving pipeline call the same feature store API — eliminating train-serve skew. Protein ML needs feature stores when:
- MSA computation is expensive (hours per sequence) and should be cached
- Multiple models share the same ESM2 embeddings
- Feature definitions change: you need to version features separately from model weights

Tools: Feast (open-source), Tecton (managed), AWS SageMaker Feature Store.

---

**Q: What is a model card and what must it include for a protein ML model?**
A: A model card (Mitchell et al., 2019) documents: (1) intended use and out-of-scope uses, (2) training data composition and exclusions, (3) performance metrics across relevant subgroups (e.g., membrane vs soluble proteins), (4) known failure modes, (5) ethical considerations. For protein ML: always specify which protein families were excluded from training — a model trained only on soluble proteins will silently fail on membrane proteins.

---

**Q: What is the difference between aleatoric uncertainty and OOD detection?**
A: Aleatoric: inherent noise in the data (measurement error). Can be captured by a model that outputs a distribution. OOD: the input is outside the training distribution — the model shouldn't be trusted at all. OOD detection uses features like Mahalanobis distance (embedding space), maximum softmax probability, or energy score to flag inputs the model has never seen.

---

**Q: What is Expected Calibration Error (ECE) and how do you fix a miscalibrated model?**
A: ECE = mean(|accuracy_in_bin - confidence_in_bin|) over confidence bins. A perfectly calibrated model has ECE=0. Fix: temperature scaling — divide logits by T before softmax. T > 1 softens predictions (reduces overconfidence). T is found by minimizing NLL on a calibration set. One of the cheapest and most effective post-hoc fixes.

---

## Key MLOps Tools Cheat Sheet

| Need | Tool | Bioinformatics Note |
|------|------|---------------------|
| Experiment tracking | MLflow, W&B | W&B has native protein structure viz |
| Data versioning | DVC, LakeFS | Track MSA files (multi-GB) |
| Model registry | MLflow, HuggingFace Hub | HF Hub for open protein models |
| Serving | FastAPI + uvicorn, TorchServe | FastAPI for prototyping; TorchServe for production |
| Containers | Docker, Singularity | Singularity for HPC (no root required) |
| Orchestration | Kubernetes, SLURM | SLURM for academic GPU clusters |
| Monitoring | Prometheus + Grafana | Monitor GPU utilisation, p99 latency |
| Feature store | Feast | Important for repeated MSA featurisation |

---
## Section 8 — Responsible AI: Model Cards, OOD Detection & Calibration

Production ML at computational biology ML teams, structural biology research labs, and Google requires more than accuracy — models must know what they don't know, and their limitations must be documented.

**Model Cards (Mitchell et al., 2019):** Standardized documentation for ML models:
- What was the training data? What was excluded?
- What populations/proteins does the model work well on? Poorly?
- Known failure modes and edge cases
- Intended vs prohibited use cases

computational biology ML teams publishes model cards for every deployed model. structural biology research labs's AlphaFold has a model card. This is non-negotiable for production deployment.

**Out-of-Distribution (OOD) Detection:** Your protein stability model was trained on soluble globular proteins. A membrane protein arrives → model confidently predicts wrong answer. OOD detection flags this.

**Calibration:** A well-calibrated model: when it says "90% confidence", it's right 90% of the time. Most neural networks are overconfident. Temperature scaling fixes this in one line.

In [ ]:
# --- Imports ---
import numpy as np              # numpy: numerical arrays and math operations
import torch                    # PyTorch: the main deep learning library
import torch.nn as nn           # nn: neural network building blocks (layers, losses)
from sklearn.calibration import calibration_curve  # calibration_curve: reliability diagram for probabilistic predictions
import matplotlib.pyplot as plt # pyplot: plotting graphs and charts

# Responsible AI: model cards, OOD detection, calibration
print("Responsible AI for Protein ML")
# Display output
print("=" * 60)

# 1. Model Card
model_card = {
    "model_name": "ProteinDDG-LoRA-v1",
    "task": "ΔΔG prediction for single-point mutations",
    "input": "Protein sequence + mutation (wt_aa, position, mut_aa)",
    "output": "ΔΔG in kcal/mol (negative=stabilizing)",
    "training_data": "SKEMPI v2 (7,085 mutations across 319 proteins)",
    "limitations": [
        "Trained on soluble proteins; may not generalize to membrane proteins",
        "Poor extrapolation to mutations > 20 residues from training distribution",
        "Does not model epistatic effects (double mutations)",
    ],
    "performance": {"Pearson_r": 0.68, "RMSE": 1.2, "test_set": "SKEMPI held-out"},
}
# Display output
print("\nModel Card:")
# --- Loop ---
for k, v in model_card.items():
    # Conditional check
    if isinstance(v, list):
        # Display output
        print(f"  {k}:"); [print(f"    - {i}") for i in v]
    else:
        # Display output
        print(f"  {k}: {v}")

# 2. Calibration check
print("\nCalibration check (regression):")
# Call torch.manual_seed
torch.manual_seed(42)
y_true = np.random.randn(200)
y_pred = y_true + np.random.randn(200) * 0.5
pred_std = np.random.uniform(0.3, 0.8, 200)  # predicted uncertainty

# Reliability: 90% CI should contain true value 90% of time
ci_90 = np.abs(y_true - y_pred) < 1.645 * pred_std
print(f"  90% CI actual coverage: {ci_90.mean():.1%} (target: 90%)")
ci_50 = np.abs(y_true - y_pred) < 0.674 * pred_std
print(f"  50% CI actual coverage: {ci_50.mean():.1%} (target: 50%)")

# 3. OOD detection via Mahalanobis distance
print("\nOOD detection (Mahalanobis distance):")
ref_features = np.random.randn(500, 10)
mu = ref_features.mean(axis=0)
sigma_inv = np.linalg.inv(np.cov(ref_features.T) + 1e-6*np.eye(10))

in_dist_sample = np.random.randn(10)
out_dist_sample = np.random.randn(10) * 3 + 5  # OOD

# --- mahalanobis() function ---
def mahalanobis(x, mu, sigma_inv):
    """Computes Mahalanobis distance between a query point and a reference distribution.
    Accounts for feature correlations via the inverse covariance matrix,
    making it more sensitive than Euclidean distance for multivariate drift detection.

    Args:
        x: 1D numpy array — the query sample (n_features,)
        mu: 1D numpy array — mean of the reference distribution (n_features,)
        inv_cov: 2D numpy array — inverse covariance matrix (n_features, n_features)
    Returns:
        float — Mahalanobis distance (larger = more out-of-distribution)"""
    diff = x - mu
    return np.sqrt(diff @ sigma_inv @ diff)

print(f"  In-distribution sample: score = {mahalanobis(in_dist_sample, mu, sigma_inv):.2f}")
print(f"  OOD sample: score = {mahalanobis(out_dist_sample, mu, sigma_inv):.2f}")
print("  Threshold: 99th percentile of calibration scores")

## Resources

### 1. Theory
- **"Designing Machine Learning Systems"** by Chip Huyen (O'Reilly, 2022) — the definitive MLOps book; covers data management, training pipelines, serving, monitoring
- **"Hidden Technical Debt in Machine Learning Systems"** (Sculley et al., Google, NeurIPS 2015): https://proceedings.neurips.cc/paper/2015/file/86df7dcfd896fcaf2674f757a2463eba-Paper.pdf
- **Continuous Delivery for Machine Learning** (Martin Fowler): https://martinfowler.com/articles/cd4ml.html
- **MLflow documentation**: https://mlflow.org/docs/latest/
- **"A Survey of Data Management for Machine Learning"** (VLDB 2021)

### 2. Must-Have Popular Resources

**Start Here (Zero Background):**
- **Made With ML** — free MLOps course by Goku Mohandas (best free curriculum): https://madewithml.com/
- **FastAPI tutorial** (official, 30 min to first API): https://fastapi.tiangolo.com/tutorial/
- **Docker for beginners** (Play With Docker labs): https://training.play-with-docker.com/
- **GitHub Actions quickstart**: https://docs.github.com/en/actions/quickstart

**Popular / Advanced:**
- **Full Stack Deep Learning** (MLOps-heavy, free): https://fullstackdeeplearning.com/
- **W&B courses** (free, with certificates): https://www.wandb.courses/
- **Chip Huyen ML Interviews Book** (systems section): https://huyenchip.com/ml-interviews-book/

### 3. Practicals
- Kaggle: "Deploy your first model with FastAPI + Docker" notebook template
- **HuggingFace Inference Endpoints** (1-click deployment): https://huggingface.co/inference-endpoints
- **W&B tutorial**: "Track your first experiment" (30 min): https://docs.wandb.ai/tutorials
- **AWS SageMaker free tier**: experiment tracking + deployment
- **DVC tutorial**: https://dvc.org/doc/start

### 4. Real-World Problems

1. **Protein structure prediction pipeline**: MLflow-tracked pipeline from sequence → ESM2 embeddings → structure classification → deployed FastAPI. Dataset: PDB sequences + SCOP labels.
2. **Drug discovery model monitoring**: Deploy ΔΔG prediction model; monitor for distribution shift when new protein targets are added. Dataset: ProteinGym substitutions.
3. **LoRA adapter serving**: Serve base ESM2 once in memory, swap per-protein-family LoRA adapters at request time. Benchmark latency vs one separate model per family.

### 5. Interview Tips
- Describe your full ML pipeline: data → features → train → eval → deploy → monitor
- For computational biology ML teams / structural biology research labs: know HPC MLOps (SLURM, not just Kubernetes), large model serving (vLLM, TGI)
- Know the difference between **online** (real-time) and **batch** inference
- **"Train-serve skew"** is the most common MLOps interview topic — know it cold
- Mention reproducibility: seeds, config files, checksums — shows production mindset
- Know: model registry vs artifact store vs feature store (they are different things)

### 6. Hot Industry Topics
- **LLM serving infrastructure**: vLLM (paged attention, KV cache), TensorRT-LLM for GPU-optimised inference
- **LoRA adapter serving**: serve one base model + thousands of adapters (parameter-efficient multi-tenant ML)
- **Protein foundation model APIs**: ESMFold API (Meta), OpenFold inference server, Chai Discovery API
- **MLOps for biology**: wet-lab-in-the-loop pipelines, uncertainty-aware deployment (don't predict confidently on OOD proteins)
- **Model cards**: documenting protein model limitations, training data, evaluation populations (computational biology ML teams publishes these for all production models)
- **FHE (Fully Homomorphic Encryption)**: run inference on encrypted patient sequences — emerging for clinical proteomics

## Timed Practice Problems

Work through these under exam conditions. Set a timer for each.

---

**Problem 1 — FastAPI endpoint validation (10 min)**

Write a Pydantic `PredictionRequest` model with these fields:
- `sequence: str` — validate it contains only standard amino acids (`ACDEFGHIKLMNPQRSTVWY`), converted to uppercase, max 1024 characters
- `wt_residue: str` — single character, must be a valid amino acid
- `mut_residue: str` — single character, must be a valid amino acid and different from `wt_residue`
- `position: int` — must be between 1 and `len(sequence)`

Write a `@validator` for each field. What error does FastAPI return if validation fails?

---

**Problem 2 — Drift detection (5 min)**

Describe three concrete ways to detect data drift in a production protein stability model. For each, state:
- What you measure
- What statistical test you use
- What action you take if drift is detected

---

**Problem 3 — MLflow experiment tracking vs model registry (3 min)**

What is the difference between:
- An **MLflow experiment** with logged runs
- The **MLflow Model Registry**

When would you promote a run from an experiment to the registry? What stages exist in the registry?

---

**Problem 4 — Docker deployment design (15 min — design question)**

Design a Docker-based deployment for a LoRA-fine-tuned ESM2 model with these requirements:
- Base ESM2-15B model is 30 GB; 5 per-family LoRA adapters are 50 MB each
- Must serve 3 protein families simultaneously
- P99 latency target: < 500 ms
- Zero-downtime deployments required

Design the Docker image(s), how adapters are loaded, and how you achieve zero-downtime updates.

---

**Problem 5 — EMA update for BYOL target network (3 min)**

BYOL (Bootstrap Your Own Latent) uses an exponential moving average (EMA) to update the target network. Implement this in 3 lines:

```python
# online_model and target_model are nn.Module instances
# tau = 0.996 (EMA decay)
# Your 3-line implementation here:
```

Hint: iterate over `zip(online_model.parameters(), target_model.parameters())`.

---

### Answers

<details>
<summary>Problem 5 answer (reveal after attempting)</summary>

```python
tau = 0.996
for online_p, target_p in zip(online_model.parameters(), target_model.parameters()):
    target_p.data = tau * target_p.data + (1 - tau) * online_p.data
```

The target network parameters are never updated by gradient descent — only by EMA. This prevents collapse in self-supervised learning.
</details>

## 🐳 Section 9 — Containerisation with Docker

Every production ML service runs in a container. Docker ensures your environment is reproducible across laptops, cloud VMs, and Kubernetes clusters.

**Why containers matter for protein ML:**
- PyTorch + CUDA versions must match exactly — Docker pins both
- Your prediction service runs identically on a T4 Colab and an A100 Lambda Labs instance
- Enables blue/green deployments and rolling updates without downtime


In [ ]:
# Dockerfile for the EGFR ΔΔG prediction FastAPI service
# Save as: Dockerfile (in project root)

dockerfile_content = '''
# ── Stage 1: base image with CUDA + Python ──
FROM pytorch/pytorch:2.2.0-cuda11.8-cudnn8-runtime

WORKDIR /app

# ── Install Python dependencies ──
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# ── Copy application code ──
COPY app/ ./app/
COPY models/ ./models/

# ── Expose port and set entrypoint ──
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
'''.strip()

print('Dockerfile content:')
print(dockerfile_content)

# Write the Dockerfile (commented out to avoid creating files in a notebook)
# with open('Dockerfile', 'w') as f:
#     f.write(dockerfile_content)

# ── Build and run commands ──
build_commands = [
    '# Build the image',
    'docker build -t protein-ml-api:v1.0 .',
    '',
    '# Run locally (CPU)',
    'docker run -p 8000:8000 protein-ml-api:v1.0',
    '',
    '# Run with GPU',
    'docker run --gpus all -p 8000:8000 protein-ml-api:v1.0',
    '',
    '# Test the endpoint',
    'curl -X POST http://localhost:8000/predict \\',
    '     -H "Content-Type: application/json" \\',
    '     -d \'{ "sequence": "MKLVIVGDGACGKT", "mutation": "L3R" }\'',
]
print('\nBuild & run commands:')
for cmd in build_commands:
    print(cmd)


## 📦 Section 10 — Data Version Control with DVC

Biological datasets change: PDB releases new structures monthly, UniProt adds sequences daily. DVC tracks dataset versions alongside model versions — so you always know which data produced which result.


In [ ]:
# DVC workflow for protein ML datasets
# Run these commands in your terminal (not in notebook)

dvc_commands = {
    'Setup': [
        'pip install dvc dvc-s3',
        'git init  # if not already a git repo',
        'dvc init',
    ],
    'Track a dataset': [
        'dvc add data/pdb_structures/',
        'git add data/pdb_structures.dvc .gitignore',
        'git commit -m "Track PDB structures dataset v1"',
    ],
    'Remote storage (S3 or Google Drive)': [
        'dvc remote add -d myremote s3://my-bucket/dvc-store',
        'dvc push',
    ],
    'Reproduce on a new machine': [
        'git clone <repo>',
        'dvc pull  # downloads the exact dataset version',
    ],
    'Track a trained model': [
        'dvc add models/pairformer_lora_v2.pt',
        'git add models/pairformer_lora_v2.pt.dvc',
        'git commit -m "Add LoRA checkpoint trained on SKEMPI v2.3"',
    ],
}

# --- Loop ---
for section, cmds in dvc_commands.items():
    # Display output
    print(f'\n# {section}')
    # --- Loop over cmds ---
    for cmd in cmds:
        # Display output
        print(f'  $ {cmd}')

# Cost estimation for cloud GPU runs
print('\n' + '='*60)
# Display output
print('Cost Estimation: Running ΔΔG Predictions at Scale')
# Display output
print('='*60)

costs = {
    'ESM-2 650M embedding (1 sequence, GPU)': ('0.5s', 'T4 @ $0.35/hr', '$0.00005'),
    'ΔΔG scan (441 mutations, GPU)':          ('3 min', 'T4 @ $0.35/hr', '$0.018'),
    'OpenFold inference (1 structure, A100)': ('8 min', 'A100 @ $1.10/hr', '$0.15'),
    'Full EGFR campaign (100 proteins)':      ('14 hrs', 'A100 @ $1.10/hr', '$15.40'),
}

# Display output
print(f'{"Task":<45} {"Time":>8} {"Instance":>18} {"Cost":>8}')
print('-' * 85)
# --- Loop ---
for task, (time, instance, cost) in costs.items():
    print(f'{task:<45} {time:>8} {instance:>18} {cost:>8}')


## Mastery Check

Before moving on, make sure you can:
1. explain how to make an ML experiment reproducible
2. explain what train-serve skew is
3. explain why drift monitoring matters in biology-facing systems
4. describe a minimal safe deployment workflow


---
## 🚀 Production-Ready ML Serving

The basic FastAPI demo earlier works for single requests. Production systems need async serving, model versioning, and performance optimization.


In [ ]:
# PRODUCTION ML SERVING — Beyond the Demo

# 1. ASYNC FASTAPI (handles concurrent requests efficiently)
ASYNC_SERVER_CODE = '''
from fastapi import FastAPI, HTTPException, BackgroundTasks  # FastAPI: async web framework; HTTPException: raise HTTP error codes; BackgroundTasks: run tasks after response
from pydantic import BaseModel, validator  # BaseModel: typed schema; validator: custom field validation decorator
import asyncio  # asyncio: Python async/await concurrency — FastAPI runs on this event loop
import torch
import numpy as np
import logging
import time
from typing import Optional, List
import hashlib

app = FastAPI(title="Protein ΔΔG Predictor", version="2.0")
logger = logging.getLogger(__name__)

# ── Model Registry ──
class ModelRegistry:
    """
    Plain English: a simple in-memory registry that maps endpoint names to loaded model versions — enables zero-downtime model swaps by updating the active model without restarting the server.

    # ---
    In production: replace this with MLflow Model Registry or a database-backed solution.
    """
    def __init__(self):
        self.models = {}  # name -> {'model': ..., 'version': ..., 'metadata': ...}
        self.active = {}  # endpoint -> model_name

    def register(self, name, model, version, metadata=None):
        self.models[name] = {'model': model, 'version': version,
                              'metadata': metadata or {}, 'loaded_at': time.time()}
        logger.info(f"Registered model {name} v{version}")

    def set_active(self, endpoint, model_name):
        if model_name not in self.models:
            raise ValueError(f"Model {model_name} not registered")
        self.active[endpoint] = model_name

    def get_active(self, endpoint):
        name = self.active.get(endpoint)
        if name is None or name not in self.models:
            raise RuntimeError(f"No active model for endpoint {endpoint}")
        return self.models[name]['model'], self.models[name]['version']

registry = ModelRegistry()

# ── Request/Response Models ──
class PredictRequest(BaseModel):
    """
    Plain English: validated request schema for the async prediction endpoint.

    # ---
    Fields:
        # Process
        sequence: protein sequence string (A-Z single-letter codes, max 500 aa)
        mutation: mutation in HGVS-like notation, e.g. "A42G" means Ala at position 42 becomes Gly
    # ---
    Note: the @validator on sequence automatically rejects unknown amino acids and sequences > 500 aa.
    """
    sequence: str
    mutation: str  # e.g., "A42G" (Ala42Gly)

    @validator('sequence')
    def validate_sequence(cls, v):
        valid = set('ACDEFGHIKLMNPQRSTVWY')
        invalid = set(v.upper()) - valid
        if invalid:
            raise ValueError(f"Invalid amino acids: {invalid}")
        if len(v) > 500:
            raise ValueError("Sequence too long (max 500 aa)")
        return v.upper()

class PredictResponse(BaseModel):
    """
    # Process
    Plain English: the typed response that the async /predict/ddg endpoint returns as JSON.

    Fields:
        # ---
        ddg:           predicted ΔΔG in kcal/mol
        uncertainty:   optional prediction uncertainty (None if model does not provide it)
        # Process
        model_version: which model version was used (for reproducibility logging)
        latency_ms:    end-to-end request latency in milliseconds
        # ---
        request_id:    short hash uniquely identifying this request (for log correlation)
    """
    ddg: float
    uncertainty: Optional[float]
    model_version: str
    latency_ms: float
    request_id: str

# ── Prediction Endpoint (async) ──
@app.post("/predict/ddg", response_model=PredictResponse)
async def predict_ddg(request: PredictRequest):
    start = time.time()
    request_id = hashlib.md5(f"{request.sequence}{request.mutation}".encode()).hexdigest()[:8]

    try:
        model, version = registry.get_active("ddg")
    except RuntimeError as e:
        raise HTTPException(status_code=503, detail=str(e))

    # Run prediction in thread pool (non-blocking for async)
    import concurrent.futures
    loop = asyncio.get_event_loop()
    with concurrent.futures.ThreadPoolExecutor() as pool:
        ddg_pred = await loop.run_in_executor(pool, model, request.sequence)

    latency_ms = (time.time() - start) * 1000
    logger.info(f"Request {request_id}: {request.mutation} → ΔΔG={ddg_pred:.3f} ({latency_ms:.1f}ms)")

    return PredictResponse(
        ddg=float(ddg_pred),
        uncertainty=0.3,  # placeholder
        model_version=version,
        latency_ms=latency_ms,
        request_id=request_id,
    )

# ── Health + Readiness Probes (required by Kubernetes) ──
@app.get("/health")
async def health():
    return {"status": "ok", "timestamp": time.time()}

@app.get("/ready")
async def ready():
    if not registry.active:
        raise HTTPException(status_code=503, detail="No model loaded")
    return {"status": "ready", "active_models": registry.active}

# ── Model Hot-Swap (A/B testing) ──
@app.post("/registry/activate")
async def activate_model(endpoint: str, model_name: str):
    try:
        registry.set_active(endpoint, model_name)
        return {"status": "activated", "endpoint": endpoint, "model": model_name}
    except ValueError as e:
        raise HTTPException(status_code=404, detail=str(e))
'''

print("ASYNC FASTAPI SERVER CODE (production-ready):")
# ---
print("Key improvements over the demo:")
# Print results
print("  1. Async endpoints: handle 100s of concurrent requests without blocking")
print("  2. Input validation with Pydantic validators")
# ---
print("  3. Model registry: register multiple versions, hot-swap without downtime")
print("  4. Health/readiness probes: required by Kubernetes orchestration")
# Print results
print("  5. Request ID + structured logging: essential for debugging in production")
print()
# ---
print("Save to: protein_server.py")
print("Run with: uvicorn protein_server:app --host 0.0.0.0 --port 8080 --workers 4")

### ONNX Export — Fast CPU Inference

ONNX (Open Neural Network Exchange) is a portable format that many inference engines can read. Exporting to ONNX and running with ONNX Runtime can give **2-10x faster CPU inference** than PyTorch, which matters for high-throughput batch predictions.

This cell exports a CNN protein predictor, benchmarks PyTorch vs ONNX Runtime side by side, and verifies that both produce identical outputs.

**Key concept:** ONNX compiles the computation graph to remove Python overhead and apply hardware-specific optimizations — the same model runs faster without changing any weights.

In [ ]:
# ONNX EXPORT — Fast CPU Inference (10-100x speedup over PyTorch)
# ONNX Runtime is the fastest way to serve PyTorch models in production

import torch                    # PyTorch: the main deep learning library
import torch.nn as nn           # nn: neural network building blocks (layers, losses)
import numpy as np              # numpy: numerical arrays and math operations
# Process
import time                     # time: measure elapsed wall-clock seconds for benchmarking

# Define a simple protein model
class ProteinPredictor(nn.Module):
    """
    Plain English: a CNN-based protein sequence model used to demonstrate ONNX export and deployment benchmarking.

    Architecture: Embedding(vocab_size, embed_dim) → Conv1d(128, k=5) → ReLU → Conv1d(64, k=3) → ReLU → AdaptiveAvgPool → Linear(64→1)
    Input:  (B, seq_len) integer amino acid indices
    Output: (B,) scalar predictions (e.g., activity score)
    """
    def __init__(self, vocab_size=21, embed_dim=64, seq_len=50):
        # Process
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.conv = nn.Sequential(
            nn.Conv1d(embed_dim, 128, 5, padding=2), nn.ReLU(),
            # Compute Conv1d(128, 64, 3, padding
            nn.Conv1d(128, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.head = nn.Linear(64, 1)

    # Process
    def forward(self, x):
        h = self.embed(x).transpose(1, 2)
        h = self.conv(h).squeeze(-1)
        return self.head(h).squeeze(-1)

# Compute model
model = ProteinPredictor()
model.eval()

# Dummy input
dummy_input = torch.randint(0, 20, (1, 50))  # (batch=1, seq_len=50)

# ── Export to ONNX ──
import os
onnx_path = '/tmp/protein_predictor.onnx'
torch.onnx.export(
    # Process
    model, dummy_input, onnx_path,
    input_names=['sequence'],
    output_names=['ddg_prediction'],
    dynamic_axes={'sequence': {0: 'batch_size'}, 'ddg_prediction': {0: 'batch_size'}},
    # Compute opset_version
    opset_version=14,
)
print(f"ONNX model saved to {onnx_path}")

# ── Benchmark: PyTorch vs ONNX Runtime ──
n_batch = 100
batch_inputs = torch.randint(0, 20, (n_batch, 50))

# PyTorch inference
t0 = time.time()
with torch.no_grad():
    for i in range(n_batch):
        # Compute _
        _ = model(batch_inputs[i:i+1])
torch_time = (time.time() - t0) * 1000 / n_batch

print(f"PyTorch inference: {torch_time:.3f} ms/sample")

try:
    # Process
    import onnxruntime as ort

    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])

    # Warm up
    for _ in range(10):
        sess.run(None, {'sequence': batch_inputs[:1].numpy()})

    # Benchmark ONNX
    t0 = time.time()
    for i in range(n_batch):
        _ = sess.run(None, {'sequence': batch_inputs[i:i+1].numpy()})
    # Compute onnx_time
    onnx_time = (time.time() - t0) * 1000 / n_batch

    print(f"ONNX Runtime inference: {onnx_time:.3f} ms/sample")
    print(f"Speedup: {torch_time/onnx_time:.1f}×")

    # Verify outputs match
    pt_out = model(batch_inputs[:1]).item()
    onnx_out = sess.run(None, {'sequence': batch_inputs[:1].numpy()})[0][0]
    print(f"Output match: PyTorch={pt_out:.6f}, ONNX={onnx_out:.6f} "
          # Compute f"(diff
          f"(diff={abs(pt_out-onnx_out):.2e})")

except ImportError:
    print("onnxruntime not installed. Install with: pip install onnxruntime")
    print()
    # Print results
    print("TYPICAL ONNX SPEEDUPS (CPU inference):")
    print("  Small MLP: 2-5×")
    print("  CNN models: 3-8×")
    print("  Transformer (small): 2-4×")
    # Print results
    print("  Key benefit: no Python overhead, compiled inference graph")
    print()
    print("PRODUCTION DEPLOYMENT STACK:")
    print("  Training: PyTorch (flexible, GPU)")
    # Print results
    print("  Export: ONNX (portable format)")
    print("  Serving: ONNX Runtime or TensorRT (fast CPU/GPU inference)")
    print("  Container: Docker + FastAPI (scalable)")
    print("  Orchestration: Kubernetes (multi-instance, auto-scaling)")

# Print results
print()
print("FULL PRODUCTION PIPELINE:")
print("  1. Train model with MLflow tracking (experiment versioning)")
print("  2. Evaluate on test set, log metrics to MLflow")
# Print results
print("  3. If test metrics pass threshold → export to ONNX")
print("  4. Register ONNX model in MLflow Model Registry with version tag")
print("  5. Deploy to FastAPI server via model registry activation")
print("  6. Monitor with KS drift detection (module 16/01)")
# Print results
print("  7. Trigger retraining when drift detected")

## GPU Memory Estimation Guide

Before deploying a protein ML model, estimate its VRAM usage to choose the right GPU.

**Rule of thumb:**
```
VRAM = param_count x bytes_per_param x multiplier
```
- **Inference:** multiplier = 2 (weights + activations)
- **Training:** multiplier = 4 (weights + gradients + optimizer states + activations)
- **float32** = 4 bytes, **float16** = 2 bytes, **bfloat16** = 2 bytes

| Model | Parameters | Inference (fp16) | Training (fp32) |
|-------|-----------|-----------------|-----------------|
| ESM-2 (8M) | 8M | ~32 MB | ~128 MB |
| ESM-2 (150M) | 150M | ~600 MB | ~2.4 GB |
| ESM-2 (650M) | 650M | ~2.6 GB | ~10.4 GB |
| ESM-2 (3B) | 3B | ~12 GB | ~48 GB |
| OpenFold3 | ~100M | ~400 MB | ~1.6 GB |

**Gradient checkpointing** trades compute for memory: ~30-40% slower but ~60% less VRAM.

In [ ]:
# GPU memory estimation and monitoring
import torch

# --- estimate_vram_gb() function ---
def estimate_vram_gb(param_count, dtype_bytes=2, multiplier=2):
    """Estimate GPU VRAM usage in GB.
    
    Args:
        param_count: number of model parameters (e.g. 650_000_000)
        dtype_bytes: 4 for float32, 2 for float16/bfloat16
        multiplier: 2 for inference, 4 for training
    Returns:
        float — estimated VRAM in GB
    """
    return param_count * dtype_bytes * multiplier / (1024**3)

# Examples
models = {
    "ESM-2 (8M)": 8_000_000,
    "ESM-2 (150M)": 150_000_000,
    "ESM-2 (650M)": 650_000_000,
    "ESM-2 (3B)": 3_000_000_000,
}

print("Model               Inference(fp16)  Training(fp32)")
print("-" * 55)
# --- Loop ---
for name, params in models.items():
    inf = estimate_vram_gb(params, dtype_bytes=2, multiplier=2)
    train = estimate_vram_gb(params, dtype_bytes=4, multiplier=4)
    print(f"{name:<20} {inf:>8.1f} GB      {train:>8.1f} GB")

# Check current GPU memory (if available)
if torch.cuda.is_available():
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    # Combine tensors
    print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print(f"Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")
    print(f"Total:     {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")
else:
    print("\nNo GPU detected — this is fine for learning, but deployment needs GPU.")
    print("See CLOUD_SETUP.md for free GPU options (Colab, Kaggle, Lightning AI).")

In [ ]:
# TODO: Implement prediction caching
prediction_cache = {}

def predict_with_cache(sequence, position, mutation):
    """Return cached result if available, otherwise compute and cache."""
    # TODO: Create a cache key from the three arguments
    cache_key = None  # replace with f"{sequence}_{position}_{mutation}"
    
    # TODO: If cache_key in prediction_cache, return the cached value
    
    # TODO: Otherwise compute result (simulate with a random number here)
    import random
    result = {"ddg": round(random.gauss(-1.0, 2.0), 3), "confidence": 0.85}
    
    # TODO: Store in cache before returning
    
    return result

# Test:
r1 = predict_with_cache("ACDEFGHIKLM", 3, "W")
r2 = predict_with_cache("ACDEFGHIKLM", 3, "W")
print(f"First call:  {r1}")
print(f"Second call: {r2}")
print(f"Same result? {r1 == r2}")  # Should be True if caching works
print(f"Cache size:  {len(prediction_cache)}")  # Should be 1

# SOLUTION:
# cache_key = f"{sequence}_{position}_{mutation}"
# if cache_key in prediction_cache:
#     return prediction_cache[cache_key]
# result = {"ddg": round(random.gauss(-1.0, 2.0), 3), "confidence": 0.85}
# prediction_cache[cache_key] = result
# return result

print("Exercise 3 ready — fill in the TODOs above")

## 🏋️ Exercise 3: Implement Prediction Caching

**Task:** The `/predict` endpoint recomputes ΔΔG every time, even for identical mutations. Add a cache.

**Hint:** Use a Python dict as cache; key = `f"{sequence}_{position}_{mutation}"`  
**Why:** Caching is one of the highest-impact MLOps optimisations — reduces latency and compute cost.

In [ ]:
# TODO: Add validation to MutationRequest
VALID_AAS = set("ACDEFGHIKLMNPQRSTVWY")

class MutationRequestValidated:
    def __init__(self, sequence: str, position: int, mutation: str):
        # TODO: Check len(sequence) >= 5, raise ValueError if not
        # TODO: Check all characters in VALID_AAS, raise ValueError with invalid chars
        # TODO: Check mutation is a single character in VALID_AAS
        self.sequence = sequence
        self.position = position
        self.mutation = mutation

# Test your validation:
try:
    bad = MutationRequestValidated(sequence="AC!", position=0, mutation="A")
    print("FAIL — should have raised ValueError")
except ValueError as e:
    print(f"PASS — caught: {e}")

try:
    short = MutationRequestValidated(sequence="AC", position=0, mutation="A")
    print("FAIL — should have raised ValueError")
except ValueError as e:
    print(f"PASS — caught: {e}")

# SOLUTION (uncomment):
# if len(sequence) < 5:
#     raise ValueError(f"Sequence too short ({len(sequence)} residues, need >= 5)")
# invalid = set(sequence.upper()) - VALID_AAS
# if invalid:
#     raise ValueError(f"Invalid amino acids: {invalid}")
# if len(mutation) != 1 or mutation.upper() not in VALID_AAS:
#     raise ValueError(f"Mutation must be a single valid amino acid, got: {mutation}")

print("Exercise 2 ready — fill in the TODOs above")

## 🏋️ Exercise 2: Add Input Validation to the FastAPI Endpoint

**Task:** The `/predict` endpoint accepts ANY sequence string. Add validation:
1. Reject sequences shorter than 5 residues
2. Reject sequences with characters not in the 20 amino acid alphabet
3. Return a clear error message for invalid inputs

**Hint:** Use a Pydantic `@validator('sequence')` or check inside the function  
**Why:** Production APIs must reject bad input before it reaches the model.

In [ ]:
# TODO: Complete this function to log Pearson r to MLflow
from scipy.stats import pearsonr
import torch.nn.functional as F

def train_with_pearson_logging(model, X_train, y_train, X_val, y_val, n_epochs=20):
    """Train model and log both MSE and Pearson r to MLflow."""
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(n_epochs):
        pred = model(X_train).squeeze()
        loss = F.mse_loss(pred, y_train)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        
        # TODO: compute val_preds on X_val (remember .detach().numpy())
        # val_preds = ???
        
        # TODO: compute pearson_r using: r, _ = pearsonr(val_preds, y_val.numpy())
        # r = ???
        
        # TODO: log both metrics with mlflow.log_metric(...)
        
        if epoch % 5 == 0:
            print(f"Epoch {epoch}: MSE={loss.item():.4f}")

# SOLUTION (uncomment to check):
# val_preds = model(X_val).squeeze().detach().numpy()
# r, _ = pearsonr(val_preds, y_val.numpy())
# mlflow.log_metric("val_mse", loss.item(), step=epoch)
# mlflow.log_metric("val_pearson_r", r, step=epoch)

print("Exercise 1 ready — fill in the TODOs above")

## 📊 Visualizing MLOps: Experiment Comparison

In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.random.seed(42)

# Simulate 5 MLflow experiment runs with different hyperparameters
runs = {
    'Run 1 (lr=1e-2)': {'epochs': range(1,21), 'val_loss': 2.5*np.exp(-0.15*np.arange(20)) + 0.8 + np.random.normal(0,0.05,20)},
    'Run 2 (lr=1e-3)': {'epochs': range(1,21), 'val_loss': 2.5*np.exp(-0.10*np.arange(20)) + 0.3 + np.random.normal(0,0.03,20)},
    'Run 3 (lr=1e-4)': {'epochs': range(1,21), 'val_loss': 2.5*np.exp(-0.05*np.arange(20)) + 0.5 + np.random.normal(0,0.02,20)},
    'Run 4 (lr=1e-3, LoRA r=4)': {'epochs': range(1,21), 'val_loss': 2.0*np.exp(-0.12*np.arange(20)) + 0.25 + np.random.normal(0,0.03,20)},
    'Run 5 (lr=1e-3, LoRA r=16)': {'epochs': range(1,21), 'val_loss': 1.8*np.exp(-0.13*np.arange(20)) + 0.22 + np.random.normal(0,0.04,20)},
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12']
for (name, data), color in zip(runs.items(), colors):
    ax1.plot(data['epochs'], data['val_loss'], '-', color=color, label=name, lw=1.5)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Validation Loss')
ax1.set_title('MLflow Experiment Tracking — Val Loss per Run', fontweight='bold')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

# Best val loss per run
best_losses = [min(d['val_loss']) for d in runs.values()]
ax2.barh(list(runs.keys()), best_losses, color=colors, edgecolor='white')
ax2.set_xlabel('Best Validation Loss')
ax2.set_title('Run Comparison — Best Val Loss', fontweight='bold')
ax2.invert_yaxis()
plt.tight_layout(); plt.show()
print(f"Best run: {list(runs.keys())[np.argmin(best_losses)]} (val_loss = {min(best_losses):.3f})")
print("This is what MLflow does — but with a web dashboard, not just matplotlib!")

## 🏋️ Exercise 1: Log a Custom Metric to MLflow

**Task:** Modify the function below to also log `val_pearson_r` (Pearson correlation between predictions and true ΔΔG) as an MLflow metric.

**Hint:** Use `mlflow.log_metric("val_pearson_r", value, step=epoch)`  
**Why:** In protein ML, Pearson r is more informative than MSE for ΔΔG regression.

## 📚 Further Reading

- **MLflow documentation** — [mlflow.org/docs/latest](https://mlflow.org/docs/latest/index.html)
- **FastAPI tutorial** — [fastapi.tiangolo.com/tutorial](https://fastapi.tiangolo.com/tutorial/)
- **Docker for Data Science** — free guide at [docker-curriculum.com](https://docker-curriculum.com/)
- **DVC documentation** — [dvc.org/doc](https://dvc.org/doc)
- **Sculley et al. (2015)** "Hidden Technical Debt in ML Systems" — DOI: 10.5555/2969442.2969519
- **Made With ML** — [madewithml.com](https://madewithml.com/) — free MLOps course with best practices

## 🎤 Interview Questions

**Q1 (Easy):** What problem does MLflow solve that git alone cannot?
<details><summary>Answer</summary>
Git tracks code changes but not experiment metadata — hyperparameters, metrics, model artifacts, and dataset versions that define a training run. MLflow logs all of these in a structured, searchable format, enabling you to compare runs, reproduce results, and trace any deployed model back to its exact training configuration. Without it, ML experiments become unreproducible even with perfect code versioning.
</details>

**Q2 (Easy):** What is an API endpoint and why would you wrap a protein model in one?
<details><summary>Answer</summary>
An API endpoint is a URL that accepts HTTP requests and returns responses. Wrapping a protein model in a REST API (e.g., POST /predict) decouples the model from client applications, allowing web apps, pipelines, and other services to get predictions without importing the model code or having GPU access. It enables serving, versioning, authentication, and scaling independently of the model's training environment.
</details>

**Q3 (Easy):** What is the purpose of Docker in ML deployment?
<details><summary>Answer</summary>
Docker packages the model, its code, dependencies, and runtime environment into a reproducible container that runs identically on any machine. This eliminates "works on my machine" problems caused by different Python versions, CUDA drivers, or library conflicts. For protein ML models that depend on specific PyTorch, BioPython, and CUDA versions, Docker ensures the production environment exactly matches the development environment.
</details>

**Q4 (Medium):** How would you detect data drift when deploying a DDG prediction model?
<details><summary>Answer</summary>
Monitor the input feature distributions using statistical tests: KS test or PSI (Population Stability Index) on ESM-2 embedding dimensions, sequence length distribution, and amino acid composition. Set up automated alerts when the distribution shift exceeds a threshold. Also monitor output drift — if predicted DDG values suddenly shift in mean or variance, the input population may have changed. For protein-specific drift, track the fraction of sequences with low pLDDT or from novel protein families not seen during training. Retrain when drift is detected and performance degrades on a labelled validation stream.
</details>

**Q5 (Medium):** Design the Pydantic schema for an API that takes a PDB file and returns binding affinity.
<details><summary>Answer</summary>
Input schema: PDB content as a string field with validation for minimum length, a chain ID field (single uppercase letter), and an optional ligand SMILES field. Output schema: predicted binding affinity (float, in kcal/mol), confidence interval (tuple of two floats), model version (string), and a warnings list for any input quality issues. Add validators to check that the PDB string contains ATOM records, the chain ID exists in the structure, and the sequence length is within the model's training range. Use Field descriptions for API documentation auto-generation.
</details>

**Q6 (Medium):** What is DVC and how does it complement git for ML projects?
<details><summary>Answer</summary>
Data Version Control (DVC) tracks large data files and model artifacts using content-addressable storage (hashing), while storing only lightweight pointer files (.dvc) in git. This lets you version datasets and models alongside code without bloating the git repository. DVC also defines reproducible ML pipelines (dvc.yaml) with explicit dependencies and outputs, enabling anyone to reproduce the full training pipeline with a single command. It supports remote storage backends (S3, GCS) for team collaboration on large datasets.
</details>

**Q7 (Medium):** How would you set up A/B testing between two protein property prediction models?
<details><summary>Answer</summary>
Deploy both models behind a load balancer or feature flag system that routes a configurable percentage of traffic to each (e.g., 90% to model A, 10% to model B). Log every prediction with the model version, input hash, prediction, and timestamp. Define success metrics upfront: prediction accuracy against eventual experimental measurements, latency, and error rate. Run for a statistically significant period (calculate required sample size based on expected effect size). Use a two-sample t-test or Mann-Whitney U test on the accuracy metrics to determine the winner. Ensure the same input always goes to the same model (sticky routing by sequence hash) to avoid confounding.
</details>

**Q8 (Hard):** Design a CI/CD pipeline for a protein ML model from training to production.
<details><summary>Answer</summary>
Trigger: git push to main. Stage 1 (CI): lint code, run unit tests on model components (data loading, preprocessing, loss computation), run a fast smoke test training (1 epoch on 100 samples) to catch shape errors. Stage 2 (Training): if tests pass, launch a full training job on GPU infrastructure (e.g., AWS SageMaker), log to MLflow, evaluate on held-out test set. Stage 3 (Validation gate): automated checks — test set performance above threshold, no data leakage detected, model size within deployment budget, latency under SLA on benchmark inputs. Stage 4 (CD): build Docker image, push to registry, deploy to staging environment, run integration tests (end-to-end API calls with known inputs/outputs). Stage 5 (Production): canary deployment (5% traffic), monitor error rates and latency for 24 hours, auto-rollback if metrics degrade, then full rollout.
</details>

**Q9 (Hard):** How would you handle model versioning when the training data changes monthly?
<details><summary>Answer</summary>
Use a semantic versioning scheme: MAJOR.MINOR.PATCH where MAJOR changes when the model architecture or features change, MINOR increments with each monthly data refresh retrain, and PATCH covers bug fixes. Track the exact dataset version (DVC hash) in the model metadata alongside the code commit hash. Maintain a model registry (MLflow Model Registry) with stages: Staging, Production, Archived. Each monthly retrain produces a new model version that must pass automated quality gates (test set performance, no regression on a golden test set of critical mutations) before promotion to Production. Keep the previous production model running in parallel for 48 hours as a rollback target. Store data lineage — which training samples, preprocessing version, and augmentation were used — so any prediction can be traced back to its training data.
</details>

**Q10 (Hard):** Estimate the cloud cost of serving ESM-2 embeddings at 1000 requests/minute.
<details><summary>Answer</summary>
ESM-2 (650M parameter model) requires approximately 2.5 GB GPU memory and takes roughly 50-100ms per sequence on an A10G GPU. At 1000 req/min (~17 req/sec), a single GPU can handle roughly 10-20 req/sec, so you need 1-2 A10G instances. AWS g5.xlarge (1x A10G) costs approximately $1.00/hour. With 2 instances for redundancy and load balancing: ~$2.00/hour or ~$1,440/month. Add costs for: load balancer ($20/month), logging and monitoring ($50/month), and data transfer ($50/month for ~100GB). For batch processing, consider spot instances at 60-70% discount. For the 3B parameter ESM-2 model, multiply GPU requirements by 4x. Caching frequently requested sequences (LRU cache keyed by sequence hash) can reduce effective GPU load by 30-50% for typical workloads.
</details>

## Notebook Complete

**You can now:**
- Set up experiment tracking with MLflow and model serving with FastAPI
- Build a CI/CD pipeline for protein ML models with monitoring and drift detection

**Where this fits:**
- Previous: 15_self_supervised_learning
- Next: 17_capstone_project/00_end_to_end_pipeline — Module 17 — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes